# Notebook 06 — Skeleton Lab

**Goal**: Interactively debug and explore the skeleton system:
1. Extract a skeleton from any FASTEXPR expression
2. Inspect field/param/group slots
3. Instantiate with custom slot values
4. Validate the result
5. Manually add skeletons to the registry
6. Load cold-start seeds from `skeleton_seeds.yaml`
7. Preview SkeletonAgent output without WQB

**No WQB quota used** — this notebook is purely local.

In [1]:
import yaml
import json
import asyncio
import nest_asyncio
nest_asyncio.apply()

from alpha_agent.config import settings
from alpha_agent.engine.skeleton_extractor import SkeletonExtractor, SkeletonTemplate
from alpha_agent.engine.validator import ExprValidator, extract_ast
from alpha_agent.knowledge.skeleton_registry import SkeletonRegistry
from alpha_agent.knowledge.vector_store import VectorStore
from alpha_agent.data.operator_kb import OperatorKB

extractor = SkeletonExtractor()

# Use a dedicated DuckDB file for skeleton lab to avoid lock conflict with alpha_memory.db
skeleton_db_path = settings.duckdb_path.with_name("skeleton_registry.db")
registry  = SkeletonRegistry(db_path=skeleton_db_path)

store     = VectorStore()
kb        = OperatorKB()
validator = ExprValidator(kb)

print("Skeleton registry DB:", skeleton_db_path)
print("Skeleton registry:", registry.stats())

/Users/weiyanguang/vscodeprojects/WorldQuant-Brain-Alpha/myenv/lib/python3.11/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:13: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange


Skeleton registry DB: /Users/weiyanguang/vscodeprojects/WorldQuant-Brain-Alpha/data/skeleton_registry.db
Skeleton registry: {'total': 11, 'active': 11, 'instances': 0}


## 1. Extract a skeleton from an expression

In [2]:
# Try any FASTEXPR expression here
TEST_EXPR = "group_rank(ts_std_dev(returns, 20) / ts_mean(volume, 20), subindustry)"

template = extractor.extract(TEST_EXPR)
if template:
    print(f"Template:   {template.template_str}")
    print(f"Operators:  {template.operators_used}")
    print(f"Field slots: {json.dumps(template.field_slots, indent=2)}")
    print(f"Param slots: {json.dumps(template.param_slots, indent=2)}")
    print(f"Group slots: {json.dumps(template.group_slots, indent=2)}")
else:
    print("Parse failed — check expression syntax")

Template:   group_rank(ts_std_dev($X1, $W1) / ts_mean($X2, $W1), $G1)
Operators:  ['group_rank', 'ts_std_dev', 'ts_mean']
Field slots: [
  {
    "name": "$X1",
    "literal": "returns",
    "semantic_hint": ""
  },
  {
    "name": "$X2",
    "literal": "volume",
    "semantic_hint": ""
  }
]
Param slots: [
  {
    "name": "$W1",
    "literal": "20",
    "type": "int_window",
    "range": [
      1,
      252
    ],
    "seen": [
      20
    ]
  }
]
Group slots: [
  {
    "name": "$G1",
    "literal": "subindustry",
    "candidates": [
      "industry",
      "market",
      "sector",
      "subindustry"
    ]
  }
]


## 2. LLM-annotate field slots (hybrid mode)

In [3]:
# This uses LLM to annotate each slot's semantic_hint (requires LLM API key)
HYPOTHESIS = "Volatility-adjusted volume signal cross-sectionally ranked within industry"

template_with_hints = asyncio.run(
    extractor.extract_with_hints(TEST_EXPR, hypothesis=HYPOTHESIS)
)

if template_with_hints:
    print("Field slots with LLM hints:")
    for slot in template_with_hints.field_slots:
        print(f"  {slot['name']} (was: {slot['literal']}) → '{slot['semantic_hint']}'")


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm.set_verbose=True'.


Provider List: https://docs.litellm.ai/docs/providers

Field slots with LLM hints:
  $X1 (was: returns) → ''
  $X2 (was: volume) → ''


## 3. Manually instantiate a skeleton

In [4]:
# For template: group_rank(ts_std_dev($X1, $W1) / ts_mean($X2, $W2), $G1)
# Substitute your own fields:

my_field_mapping = {
    "$X1": "close",
    "$X2": "turnover",
}
my_param_mapping = {
    "$W1": 30,
    "$W2": 30,
}
my_group_mapping = {
    "$G1": "industry",
}

if template:
    concrete_expr = SkeletonExtractor.instantiate(
        template.template_str,
        field_mapping=my_field_mapping,
        param_mapping=my_param_mapping,
        group_mapping=my_group_mapping,
    )
    print(f"Instantiated: {concrete_expr}")
    result = validator.validate(concrete_expr)
    print(f"Validation:   {result}")

Instantiated: group_rank(ts_std_dev(close, 30) / ts_mean(turnover, 30), industry)
Validation:   PASS


## 4. Test isomorphism (two expressions → same skeleton?)

In [5]:
# Two structurally identical expressions with different fields/params
# should produce the same template_str

EXPR_A = "group_rank(ts_std_dev(returns, 20) / ts_mean(volume, 20), subindustry)"
EXPR_B = "group_rank(ts_std_dev(close, 30) / ts_mean(turnover, 30), industry)"

tmpl_a = extractor.extract(EXPR_A)
tmpl_b = extractor.extract(EXPR_B)

print(f"Template A: {tmpl_a.template_str if tmpl_a else 'None'}")
print(f"Template B: {tmpl_b.template_str if tmpl_b else 'None'}")
print(f"Isomorphic: {tmpl_a is not None and tmpl_b is not None and tmpl_a.template_str == tmpl_b.template_str}")

# Different structure should NOT be isomorphic
EXPR_C = "rank(ts_std_dev(returns, 20))"
tmpl_c = extractor.extract(EXPR_C)
print(f"\nTemplate C: {tmpl_c.template_str if tmpl_c else 'None'}")
print(f"A == C: {tmpl_a is not None and tmpl_c is not None and tmpl_a.template_str == tmpl_c.template_str}")

Template A: group_rank(ts_std_dev($X1, $W1) / ts_mean($X2, $W1), $G1)
Template B: group_rank(ts_std_dev($X1, $W1) / ts_mean($X2, $W1), $G1)
Isomorphic: True

Template C: rank(ts_std_dev($X1, $W1))
A == C: False


## 5. Load cold-start seeds from skeleton_seeds.yaml

In [6]:
from pathlib import Path

seeds_path = Path("../alpha_agent/data/assets/skeleton_seeds.yaml")
with open(seeds_path) as f:
    seeds_data = yaml.safe_load(f)

print(f"Found {len(seeds_data['skeletons'])} seed skeletons in YAML")
for s in seeds_data['skeletons']:
    print(f"  {s['template_str'][:70]}")

Found 10 seed skeletons in YAML
  group_rank(($X1 - $X2) / $X2, $G1)
  group_rank(ts_std_dev($X1, $W1), $G1)
  group_rank(ts_std_dev($X1, $W1) / ts_mean($X2, $W2), $G1)
  ts_corr($X1, abs($X2), $W1)
  group_rank(($X1 - ts_mean($X1, $W1)) / ts_std_dev($X1, $W1), $G1)
  trade_when(ts_rank(ts_std_dev($X1, $W1), $W2) < $W3, $X2, -1)
  group_rank(ts_corr($X1, $X2, $W1), $G1)
  regression_neut($X1, ts_std_dev($X2, $W1))
  if_else(rank($X1) > $W1, $X2, -1 * $X2)
  group_neutralize(rank($X1), bucket(rank(cap), range='0.1,1,0.1'))


In [7]:
# Ingest all seeds into the registry (idempotent — safe to re-run)
loaded = 0
for seed in seeds_data['skeletons']:
    sk_id = registry.upsert(
        template_str=seed['template_str'],
        operators_used=seed.get('operators_used', []),
        field_slots=seed.get('field_slots', []),
        param_slots=seed.get('param_slots', []),
        group_slots=seed.get('group_slots', []),
        origin_hypothesis=seed.get('origin_hypothesis', ''),
    )
    loaded += 1
    print(f"  Upserted [{sk_id[:8]}] {seed['template_str'][:50]}")

print(f"\nLoaded {loaded} seeds. Registry now: {registry.stats()}")

  Upserted [2a6d5b37] group_rank(($X1 - $X2) / $X2, $G1)
  Upserted [3128cf7d] group_rank(ts_std_dev($X1, $W1), $G1)
  Upserted [42aab681] group_rank(ts_std_dev($X1, $W1) / ts_mean($X2, $W2
  Upserted [7c625474] ts_corr($X1, abs($X2), $W1)
  Upserted [e7c5d916] group_rank(($X1 - ts_mean($X1, $W1)) / ts_std_dev(
  Upserted [dfe885dd] trade_when(ts_rank(ts_std_dev($X1, $W1), $W2) < $W
  Upserted [fded42e8] group_rank(ts_corr($X1, $X2, $W1), $G1)
  Upserted [05433f5d] regression_neut($X1, ts_std_dev($X2, $W1))
  Upserted [4acf7ee4] if_else(rank($X1) > $W1, $X2, -1 * $X2)
  Upserted [bfda36dd] group_neutralize(rank($X1), bucket(rank(cap), rang

Loaded 10 seeds. Registry now: {'total': 11, 'active': 11, 'instances': 0}


## 6. Preview SkeletonAgent output (no WQB)

In [8]:
from alpha_agent.engine.skeleton_agent import SkeletonAgent

# Set common fields so validator can check
COMMON_FIELDS = {
    "close", "open", "high", "low", "volume", "returns", "vwap",
    "turnover", "cap", "sharesout", "adv20", "pbRatio", "peRatio",
    "dividendYield", "netIncome", "totalAssets",
}

agent = SkeletonAgent(
    skeleton_registry=registry,
    vector_store=store,
    known_fields=COMMON_FIELDS,
)

variants = asyncio.run(
    agent.generate(dataset="pv1", k_per_skeleton=3, max_seeds=3)
)

print(f"Generated {len(variants)} skeleton variants:\n")
for v in variants:
    print(f"  Skeleton: {v.skeleton_id[:8]}")
    print(f"  Expr:     {v.expression}")
    print(f"  Fields:   {v.field_mapping}")
    print(f"  Params:   {v.param_mapping}")
    result = validator.validate(v.expression)
    print(f"  Valid:    {result.ok}  {result.warnings if result.warnings else ''}")
    print()


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm.set_verbose=True'.


Provider List: https://docs.litellm.ai/docs/providers

[SkeletonAgent] LLM call failed: APIError: DeepseekException - deepseek does not support parameters: {'response_format': {'type': 'json_object'}}, for model=deepseek-reasoner. To drop these, set `litellm.drop_params=True` or for proxy:

`litellm_settings:
 drop_params: true`


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm.set_verbose=True'.


Provider List: https://docs.litellm.ai/docs/providers

[SkeletonAgent] LLM call failed: APIError: DeepseekException - deepseek does not support parameters: {'response_format': {'type': 'json_object'}}, for model=deepseek-reasoner. To drop these, set `litellm.drop_params=True` or for proxy:

`litellm_settings:
 drop_params: true`


Give Feedback / Get Help: h

## 7. Manually add or edit a skeleton

In [9]:
# Add a custom skeleton to the registry
CUSTOM_TEMPLATE = "ts_rank(ts_mean($X1, $W1), $W2)"

custom_id = registry.upsert(
    template_str=CUSTOM_TEMPLATE,
    operators_used=["ts_rank", "ts_mean"],
    field_slots=[
        {"name": "$X1", "literal": "?", "semantic_hint": "financial signal to rank time-series"}
    ],
    param_slots=[
        {"name": "$W1", "literal": "5", "type": "int_window", "range": [5, 60], "seen": [5]},
        {"name": "$W2", "literal": "252", "type": "int_window", "range": [60, 252], "seen": [252]},
    ],
    group_slots=[],
    origin_hypothesis="Time-series rank of rolling mean — captures mean-reversion speed",
)
print(f"Added custom skeleton: {custom_id[:8]}")
print(f"Registry now: {registry.stats()}")

Added custom skeleton: 74991afd
Registry now: {'total': 11, 'active': 11, 'instances': 0}


In [ ]:
# Optional: release DB/file handles before switching notebooks
import gc

for obj_name in ["memory", "registry", "store", "pi"]:
    obj = globals().get(obj_name)
    close_fn = getattr(obj, "close", None)
    if callable(close_fn):
        try:
            close_fn()
            print(f"Closed: {obj_name}")
        except Exception as e:
            print(f"Failed to close {obj_name}: {e}")

for obj_name in ["memory", "registry", "store", "pi"]:
    if obj_name in globals():
        del globals()[obj_name]

gc.collect()
print("Released DB/file handles and cleared related globals.")

## ✅ Notebook 06 Complete

You can now:
- Extract and inspect skeletons from any FASTEXPR string
- Manually instantiate templates with custom slot values
- Verify structural isomorphism between expressions
- Load cold-start seeds from `skeleton_seeds.yaml`
- Preview SkeletonAgent output locally before spending WQB quota

Use this notebook to debug new skeleton designs before running the full pipeline.